# Suite YML — YAML

YAML is a thin encoding of Plain data. Loading follows the YAML 1.2 core schema; dumping quotes anything a YAML 1.1
or 1.2 reader would misread. Cases cover scalar resolution, quoting safety against a stock YAML 1.1 loader, line
breaks and control characters, rejection rules, and graph round trips.

In [ ]:
import math
import sys

import yaml

from mbse.Schemas.Framework import Plain, Proxies, Schemas as S, YAML
from mbse.Schemas.Framework.Errors import DecodeError
from support import raises, same_graph, text

Item = S.OfObject.Builder().properties(text("s"), text("i", int), text("f", float), text("b", bool),
                                       text("raw", bytes)).create()
Link = S.OfRelation.Builder().links("from", "to").create()
Item = S.OfObject.Builder(Item).relations(lambda r: r.name("out").of(Link).me("from"),
                                          lambda r: r.name("in").of(Link).me("to")).update()
Proxies.register("Item", Item)
Proxies.register("Link", Link)
B = Proxies.Builders
FromYAML = YAML.FromYAML(B)


def scalar(text_):
    return YAML.loads(f"x: {text_}")["x"]

## YML-01 · Scalar resolution follows the YAML 1.2 core schema

In [ ]:
expected = {
    "~": None, "null": None, "Null": None, "NULL": None, "": None,
    "true": True, "True": True, "TRUE": True, "false": False, "False": False, "FALSE": False,
    "yes": "yes", "no": "no", "on": "on", "off": "off", "y": "y", "n": "n", "Yes": "Yes",
    "0": 0, "-0": 0, "+1": 1, "010": 10, "0o17": 15, "0x1F": 31, "0x1f": 31, "-12": -12,
    "0b101": "0b101", "1_000": "1_000", "0o19": "0o19", "0xZZ": "0xZZ",
    "1.5": 1.5, ".5": 0.5, "1.": 1.0, "1e3": 1000.0, "1E3": 1000.0, "-1.5e-3": -0.0015, "+.inf": math.inf,
    ".inf": math.inf, "-.Inf": -math.inf, ".INF": math.inf,
    "2026-09-01": "2026-09-01", "2026-09-01T09:00:00Z": "2026-09-01T09:00:00Z", "1:30": "1:30",
    "12:30:45": "12:30:45", "nulls": "nulls", "truely": "truely", "=": "=",
}
for source, value in expected.items():
    got = scalar(source)
    assert got == value and type(got) is type(value), (source, got)
for source in [".nan", ".NaN", ".NAN"]:
    assert math.isnan(scalar(source))

## YML-02 · Strings that readers misread are quoted: both our loader and stock YAML 1.1 read them back

In [ ]:
tricky = ["yes", "No", "on", "OFF", "y", "true", "null", "~", "", "010", "0o17", "0x1F", "1e3", "1_000", "1:30",
          "2026-09-01", ".inf", ".nan", "NaN", "Infinity", "-0.0", "+1", " lead", "trail ", "a: b", "- dash",
          "#hash", "x #y", '"dq"', "'sq'", "@at", "`tick", "%pct", "!bang", "&amp", "*star", "|", ">", "{}", "[]",
          ",", "?", "<<", "---", "...", "- ", ": ", "a\nb", "\t", "\x00", "\x07\x7f\x9b", "\ufeffbom", "日本語 🚀"]
for value in tricky:
    dumped = YAML.dumps({"v": value})
    assert YAML.loads(dumped) == {"v": value}, (value, dumped)
    assert yaml.safe_load(dumped) == {"v": value}, (value, dumped)

## YML-02b · YAML 1.1 single-letter booleans are quoted (Finding F14, fixed)
The YAML 1.1 specification makes `y`, `Y`, `n` and `N` booleans, but PyYAML's resolver omits them, so the checks
above (which use PyYAML as the 1.1 reader) could not see that they were written unquoted. The cross-implementation
conformance suite caught it: a spec-compliant 1.1 reader read `y` as true.

In [ ]:
for value in ["y", "Y", "n", "N"]:
    assert YAML.dumps({"v": value}) == f"v: '{value}'\n"

## YML-03 · Finding F1 (fixed): Unicode line breaks survive (NEL, LS, PS)
PyYAML wrote NEL raw inside single quotes, where it folds into a space. Such strings are now double-quoted.

In [ ]:
for value in ["a\x85b", "\x85", "a\u2028b", "a\u2029b", "a\nb\x85c", "\r\n", "a\rb", "x\n", " \n y \n "]:
    dumped = YAML.dumps({"v": value})
    assert YAML.loads(dumped)["v"] == value and yaml.safe_load(dumped)["v"] == value, (value, dumped)
assert YAML.dumps({"v": "a\x85b"}) == 'v: "a\\Nb"\n'

## YML-04 · Long, multi-line and whitespace-heavy strings keep their exact text

In [ ]:
for value in ["word " * 200, "x" * 5000, "line\n" * 50, "  indented\n    more\n", "\t\ttabs\t", " " * 10]:
    assert YAML.loads(YAML.dumps({"v": value}))["v"] == value

## YML-05 · Numbers keep their exact values; non-finite floats go through Plain's strings

In [ ]:
for value in [0, -1, 2**64, 2**200, -(2**100)]:
    assert FromYAML(Item, YAML.ToYAML(Item, B.Item().i(value).create())).i == value
for value in [0.1, 1 / 3, 1e308, 5e-324, -0.0, 1e-7, 123456789.125]:
    back = FromYAML(Item, YAML.ToYAML(Item, B.Item().f(value).create())).f
    assert back == value and math.copysign(1, back) == math.copysign(1, value)
for value in [math.inf, -math.inf, math.nan]:
    out = YAML.ToYAML(Item, B.Item().f(value).create())
    assert ".inf" not in out and ".nan" not in out  # written as the strings Plain uses
    back = FromYAML(Item, out).f
    assert (math.isnan(back) and math.isnan(value)) or back == value

## YML-06 · Rejected input: the framework's YAML rules, with the same reason and position in every binding
One document; string keys; no duplicate keys; no tags but `!`, `!!str`, `!!seq` and `!!map`; no undefined aliases.
If there are several problems, the first in the text is reported. The table is shared verbatim with the TypeScript
suite; rows are input and the expected message, or None when the input is accepted.

In [ ]:
import json as stdlib_json

RULES = stdlib_json.loads(r'''[["a: 1\na: 2", "line 2, column 1: duplicate key 'a'"], ["x:\n  a: 1\n  a: 2", "line 3, column 3: duplicate key 'a'"], ["a: {b: 1, b: 2}", "line 1, column 11: duplicate key 'b'"], ["---\na: 1\n---\nb: 2", "line 3, column 1: expected a single document, found another"], ["a: 1\n---\nb: 2", "line 2, column 1: expected a single document, found another"], ["1: x", "line 1, column 1: keys must be strings, got int 1"], ["null: 1", "line 1, column 1: keys must be strings, got NoneType None"], ["true: 1", "line 1, column 1: keys must be strings, got bool True"], ["1.5: x", "line 1, column 1: keys must be strings, got float 1.5"], ["? [a]\n: 1", "line 1, column 3: keys must be strings, got a sequence"], ["? {a: 1}\n: 1", "line 1, column 3: keys must be strings, got a mapping"], ["a:\n  2: x", "line 2, column 3: keys must be strings, got int 2"], ["!!binary AAAA", "line 1, column 1: unsupported tag !!binary"], ["a: !!binary AAAA", "line 1, column 4: unsupported tag !!binary"], ["!!set {a, b}", "line 1, column 1: unsupported tag !!set"], ["a: !!timestamp 2026-01-01", "line 1, column 4: unsupported tag !!timestamp"], ["a: !!python/object:os.system x", "line 1, column 4: unsupported tag !!python/object:os.system"], ["a: !custom x", "line 1, column 4: unsupported tag !custom"], ["a: !!int 1", "line 1, column 4: unsupported tag !!int"], ["a: !!omap []", "line 1, column 4: unsupported tag !!omap"], ["a: [*x]", "line 1, column 5: found undefined alias 'x'"], ["a: 1\na: 2\nb: [", "line 2, column 1: duplicate key 'a'"], ["a: !!int x\nb: {", "line 1, column 4: unsupported tag !!int"], ["\u00e9\ud83d\ude80: 1\n\u00e9\ud83d\ude80: 2", "line 2, column 1: duplicate key '\u00e9\ud83d\ude80'"], ["a: &x 1\nb: !!float 2", "line 2, column 4: unsupported tag !!float"], ["\u00e9\ud83d\ude80: !!int 1", "line 1, column 5: unsupported tag !!int"], ["a: ! 12", null], ["a: !!str 010", null], ["!!map {a: 1}", null], ["!!seq [1]", null], ["a: !!str\n  x", null], ["k: &a !!str v\nj: *a", null], ["a: \u0001", "line 1, column 4: unacceptable character #x0001: special characters are not allowed"], ["a: b\n# \u007f", "line 2, column 3: unacceptable character #x007f: special characters are not allowed"], ["k: \ud800", "line 1, column 4: unacceptable character #xd800: special characters are not allowed"], ["\u00e9\ud83d\ude80: \u001b", "line 1, column 5: unacceptable character #x001b: special characters are not allowed"], ["?\n", "line 1, column 2: keys must be strings, got NoneType None"], ["a: &k b\n*k : 1\n", null], ["b: 1\nc: &k b\n*k : 2\n", "line 3, column 1: duplicate key 'b'"], ["a: &x !!binary AAAA", "line 1, column 4: unsupported tag !!binary"], ["a: !!binary &x AAAA", "line 1, column 4: unsupported tag !!binary"], ["{a: 1, a: 2", "line 1, column 8: duplicate key 'a'"], ["a: 1\r\na: 2", "line 2, column 1: duplicate key 'a'"], ["&a b: 1\n&c b: 2", "line 2, column 1: duplicate key 'b'"], ["? !!str\n: 1\n? !!str\n: 2", "line 3, column 3: duplicate key ''"], ["a: 1\na: 2\n1: x", "line 2, column 1: duplicate key 'a'"], ["{1: x, a: 1, a: 2}", "line 1, column 2: keys must be strings, got int 1"], ["[*x, *y]", "line 1, column 2: found undefined alias 'x'"]]''')
assert len(RULES) == 48
for source, message in RULES:
    try:
        YAML.loads(source)
    except DecodeError as error:
        assert str(error) == message, (source, str(error))
    else:
        assert message is None, source
assert YAML.loads("a: ! 12") == {"a": "12"}  # the non-specific tag makes a string, as YAML 1.2 says
assert YAML.loads("a: !!str 010") == {"a": "010"}

## YML-06b · Syntax errors: one line, with a position; the reason comes from the binding's parser

In [ ]:
for bad in ["a: [unclosed", "a: 'open", "\t- tab indent", "a: b: c", "a: \x01"]:
    try:
        YAML.loads(bad)
    except DecodeError as error:
        assert error.line is not None and error.column is not None and "\n" not in str(error), (bad, str(error))
        assert str(error) == f"line {error.line}, column {error.column}: {error.reason}"
    else:
        raise AssertionError(bad)
with raises(DecodeError, match="input is not valid utf-8"):
    YAML.loads(b"a: \xff")

## YML-07 · Accepted YAML features that map onto plain data (pinned)
Anchors and aliases to plain values are fine. The `<<` merge key is a YAML 1.1 feature, so it loads as an ordinary
key. An empty document loads as null.

In [ ]:
assert YAML.loads("a: &x 1\nb: *x") == {"a": 1, "b": 1}
assert YAML.loads("base: &b {k: 1}\nd:\n  <<: *b\n  j: 2")["d"] == {"<<": {"k": 1}, "j": 2}
assert YAML.loads("") is None and YAML.loads("# only a comment\n") is None
assert YAML.loads("[1, 'two', {three: 3}]") == [1, "two", {"three": 3}]
assert YAML.loads(b"a: 1") == {"a": 1}
with raises(DecodeError, match="$: expected an object snapshot"):
    FromYAML(Item, "")  # an empty document is not a snapshot

## YML-08 · Dumping: no aliases, key order kept, Unicode kept

In [ ]:
shared = {"k": 1}
dumped = YAML.dumps({"b": shared, "a": shared, "日": "本"})
assert "&" not in dumped and "*" not in dumped
assert dumped.index("b:") < dumped.index("a:") and "日: 本" in dumped
assert YAML.dumps(5) == "5\n...\n" and YAML.loads(YAML.dumps(5)) == 5

## YML-09 · Hand-written snapshots: 1.2 rules apply, and native types stay strict

In [ ]:
written = "root: s0\nobjects:\n  s0:\n    s: 2026-09-28\n    i: 010\n    f: .inf\n    b: true\n"
item = FromYAML(Item, written)
assert (item.s, item.i, item.f, item.b) == ("2026-09-28", 10, math.inf, True)
with raises(DecodeError, match="$.objects.s0.b: expected bool, got str"):
    FromYAML(Item, "root: s0\nobjects:\n  s0:\n    b: yes\n")  # 'yes' is a string
with raises(DecodeError, match="$.objects.s0.f: expected float, got int"):
    FromYAML(Item, "root: s0\nobjects:\n  s0:\n    f: 1\n")  # 1 is an int, not a float
with raises(DecodeError, match="$.objects.s0.i: expected int, got float"):
    FromYAML(Item, "root: s0\nobjects:\n  s0:\n    i: 1.0\n")

## YML-10 · YAML agrees with Plain, and graphs round trip

In [ ]:
hub = B.Item().s("hub").raw(b"\x00\xff").out(lambda x: x.to(lambda y: y.s("yes"))).out(
    lambda x: x.to(lambda y: y.s("010"))).create()
graph = Plain.ToPlain.Reachable(Item, hub)
text_ = YAML.ToYAML.Reachable(Item, hub)
assert YAML.loads(text_) == graph and yaml.safe_load(text_) == graph
back = FromYAML.Reachable(Item, text_)
assert back is not hub and same_graph(Plain.ToPlain.Reachable(Item, back), graph)
with raises(DecodeError, match="$.objects.s0.out[0].to: unresolved reference 's1'"):
    FromYAML(Item, YAML.ToYAML(Item, hub))
assert FromYAML.OfNative(S.OfNative.Data(str), YAML.ToYAML.OfNative(S.OfNative.Data(str), "yes")) == "yes"

## YML-11 · Without PyYAML the package still imports, and YAML use fails with a clear message

In [ ]:
import importlib

saved = sys.modules.get("yaml")
try:
    sys.modules["yaml"] = None  # make `import yaml` fail
    YAML._yaml.cache_clear()
    with raises(ImportError, match="requires PyYAML"):
        YAML.dumps({"a": 1})
    import mbse.Schemas.Framework as framework
    importlib.reload(framework)  # the package itself does not need PyYAML
finally:
    sys.modules["yaml"] = saved
    YAML._yaml.cache_clear()
assert YAML.loads("a: 1") == {"a": 1}

## YML-13 · Every shape of plain data round-trips; only plain data can be written

In [ ]:
shapes = [
    [],
    {},
    [1, "a", None, True, 1.5, [], {}],
    [[1, [2, [3]]], [{"k": [{}]}], {"a": []}],
    {"a": {"b": {"c": [1, {"d": None}]}}, "e": [], "f": {}},
    {"nan": math.nan, "inf": math.inf, "ninf": -math.inf, "big": 1e16, "tiny": 5e-324, "neg": -0.0},
    [["x"], [["y"]]],
]
for shape in shapes:
    dumped = YAML.dumps(shape)
    back, back11 = YAML.loads(dumped), yaml.safe_load(dumped)
    if isinstance(shape, dict) and "nan" in shape:
        assert math.isnan(back["nan"]) and math.isnan(back11["nan"])
        back.pop("nan"), back11.pop("nan")
        shape = {k: v for k, v in shape.items() if k != "nan"}
        assert math.copysign(1, back["neg"]) == -1
    assert back == shape and back11 == shape, (shape, dumped)
assert YAML.dumps({"x": 1e16}) == "x: 1.0e+16\n"  # a float keeps a '.' so every reader sees a float
with raises(TypeError, match="keys must be str, not int"):
    YAML.dumps({1: 2})
with raises(TypeError, match="keys must be str, not int"):
    YAML.dumps([{"a": {1: 2}}])
for value, name in [(b"x", "bytes"), ({1, 2}, "set"), (object(), "object")]:
    with raises(TypeError, match=f"cannot represent {name} as a YAML scalar"):
        YAML.dumps({"a": [value]})